# 🏥 Build a Clinical Knowledge RAG — Hands-on Guide

**Anaconda Desktop · FAISS · FastAPI — a clinical RAG that runs entirely on your own machine.**

> **Owner:** Federico Rubiano ([@federicorubiano](https://github.com/federicorubiano)) | **Status:** Tested end-to-end (2026-06-17) | **Estimated time:** 60–90 minutes

---

> ## ⚠️ MEDICAL DISCLAIMER
>
> **This notebook is for educational and demonstration purposes only.** Nothing it produces — generated text, citations, or clinical summaries — constitutes medical advice, diagnosis, or treatment. Always consult a qualified, licensed healthcare professional before any clinical decision. Do not use this tool in real patient care. The authors and contributors accept no liability for decisions made on the basis of this system's output.

---

## Audience

Python developers and data scientists who want to build a retrieval-augmented generation (RAG) system end to end and keep every part of it — documents, embeddings, and the language model — running locally. No prior RAG experience needed.

## What you'll learn

By the end you will be able to:

1. **Build** a FAISS dense-vector index from a scraped corpus using a local embedding model.
2. **Retrieve** the most relevant passages for a question with instruction-following embeddings.
3. **Generate** grounded, citation-enforced answers from a locally-served chat model.
4. **Serve** the pipeline through FastAPI and **evaluate** answer quality with a reproducible harness.

## Why run it locally

Everything here runs on your own machine — the documents you index, the questions you ask, and the models that answer them. Nothing is sent to a cloud API, so sensitive information never leaves the environment you control. That makes this approach a fit for regulated or air-gapped settings like healthcare and legal. The models and libraries come from Anaconda Desktop's catalog and the Anaconda `main` channel, which keeps the whole stack reproducible — and it all runs on your machine.

## Prerequisites

**Knowledge:** comfortable running Python; basic conda environments. RAG is taught here from scratch.

**Installed & running:** Anaconda Desktop, the project conda env (`environment.yml`), both model servers started with `scripts/serve_models.sh`, and the index built (`python scripts/build_index.py`). See the README for full setup. Anaconda Desktop requires an Anaconda account to sign in (organization users use their assigned credentials).

**Hardware:** ~32 GB RAM recommended — the default models use roughly 22 GB with both servers running. On a smaller machine (e.g., 16 GB), point `INFERENCE_MODEL` / `EMBEDDING_MODEL` at smaller models from the Desktop catalog (see the README); note that changing the embedding model means rebuilding the index.

**Dependencies:** Anaconda Desktop — required (the models run here; there's no cloud fallback). Merck Manual website — used only for scraping; if it's unreachable, a sample corpus in `data/raw/` is the fallback.

> ℹ️ **Two servers, side by side.** `scripts/serve_models.sh` launches both the embedding model and the chat model — each on its own port — and writes their URLs into `.env`. They run concurrently; there's no swapping between them.

> ▶ **Before you run this notebook** — three things, or the cells below will fail:
> 1. **Start both model servers** — from the repo root: `bash scripts/serve_models.sh` (it writes their ports into `.env`).
> 2. **Launch Jupyter from inside the repo** so paths resolve, e.g. `cd <repo> && conda activate anaconda-clinical-rag && jupyter lab`.
> 3. **Select the `anaconda-clinical-rag` kernel** (Kernel → Change Kernel) so the project's packages import.
>
> If you stop or restart the servers their ports change — re-run `serve_models.sh` **and** restart the kernel (Kernel → Restart) before re-running cells.


## Section 0 — Environment check

*Start state: you've created and activated the project conda env. Here we confirm the project's packages import before building anything.*

In [ ]:
import sys, os

# Locate the repo root by walking up from the kernel's working directory to the
# folder that holds the project markers. Works whether Jupyter's cwd is the repo
# root or the notebooks/ subfolder — no hardcoded paths.
def _find_root(start):
    p = os.path.abspath(start)
    while True:
        if os.path.isdir(os.path.join(p, 'scripts')) and os.path.exists(os.path.join(p, 'environment.yml')):
            return p
        if os.path.dirname(p) == p:
            return None
        p = os.path.dirname(p)

project_root = _find_root(os.getcwd())
if project_root is None:
    raise RuntimeError(
        'Could not find the project root from cwd=' + repr(os.getcwd()) + '. '
        'Launch Jupyter from inside the repo and select the anaconda-clinical-rag kernel.'
    )
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')
print(f"Python: {sys.version.split()[0]}")


In [ ]:
# Verify the project's packages import cleanly
import importlib

packages = {
    'faiss': 'faiss-cpu',
    'fastapi': 'fastapi',
    'gradio': 'gradio',
    'evidently': 'evidently',
    'requests': 'requests',
    'numpy': 'numpy',
    'dotenv': 'python-dotenv',
    'tqdm': 'tqdm',
}

print('Package status:')
for module, pkg in packages.items():
    try:
        m = importlib.import_module(module)
        print(f"  ✅  {pkg:<16} {getattr(m, '__version__', 'installed')}")
    except ImportError:
        print(f"  ❌  {pkg:<16} NOT INSTALLED")

**✅ Checkpoint 0:** every package shows a green check — your environment is ready.

Expected output (versions will vary):
```text
Package status:
  ✅  faiss-cpu        1.14.x
  ✅  fastapi          0.135.x
  ✅  gradio           6.14.x
  ✅  evidently        0.7.x
  ✅  requests         2.34.x
  ✅  numpy            1.26.x
  ✅  python-dotenv    1.x
  ✅  tqdm             4.68.x
```

## Section 1 — How the pipeline works

The RAG flow you're about to build:

```
User question
    │
    ▼
DenseRetriever
    └─ embedding model (query instruction prefix)
         → FAISS dense search → top-k chunks
    │
    ▼
DesktopClient (local chat model)
    │   System prompt: citation rules + structure
    │   User prompt:   retrieved context + question
    ▼
Answer + Citations + Medical Disclaimer
    └─ served by FastAPI /query
```

This guide uses **Qwen2.5-14B-Instruct** for chat and **Qwen3-Embedding-8B** for embeddings, both served locally by Anaconda Desktop.

**Why this design?** Dense retrieval finds passages by *meaning*, not keywords, and the embedding model's instruction-following design ranks query/document relevance well enough that no separate reranking step is needed — fewer moving parts, all local.

## Section 2 — Build the index (embeddings)

*Start state: both model servers are running (`scripts/serve_models.sh`).*

We scrape the Merck Manual (if needed) and embed every chunk into a FAISS index. If `data/index/merck.faiss` already exists, this cell just confirms it.

In [ ]:
import subprocess, os

index_path = os.path.join(project_root, 'data', 'index', 'merck.faiss')
raw_dir = os.path.join(project_root, 'data', 'raw')

if not os.path.exists(index_path):
    print('No index found — scraping + building (this calls the Desktop embedding server)...')
    # Tip: you can also run these in a terminal:
    #   python scripts/scraper.py
    #   python scripts/build_index.py
    subprocess.run([sys.executable, os.path.join(project_root, 'scripts', 'scraper.py')], check=False)
    subprocess.run([sys.executable, os.path.join(project_root, 'scripts', 'build_index.py')], check=False)
else:
    print('Index already present — skipping rebuild.')

print('\nraw topics:', len([f for f in os.listdir(raw_dir) if f.endswith('.txt')]) if os.path.isdir(raw_dir) else 0)
print('index exists:', os.path.exists(index_path))

**✅ Checkpoint 2:** `data/index/merck.faiss` and `data/index/chunks.json` exist and a chunk count is reported.

Expected output (numbers will vary):
```text
Index already present — skipping rebuild.

raw topics: 7
index exists: True
```

> ℹ️ If indexing errors out, see **README → Known issues** for current workarounds.

## Section 3 — Retrieve relevant passages

*Start state: the index exists (Section 2) and both servers are running. We load the retriever and run a query — retrieval uses the embedding model to turn your question into a vector.*

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

from src.retriever import DenseRetriever  # (HybridRetriever is a backwards-compatible alias)

_desktop = os.getenv('DESKTOP_API_URL', 'http://localhost:8080/v1')
retriever = DenseRetriever(
    faiss_path=os.path.join(project_root, 'data/index/merck.faiss'),
    chunks_path=os.path.join(project_root, 'data/index/chunks.json'),
    api_url=os.getenv('EMBEDDING_API_URL', _desktop),
    embedding_model=os.getenv('EMBEDDING_MODEL', 'Qwen3-Embedding-8B'),
    top_k=int(os.getenv('TOP_K', 5)),
)
print(f'Retriever loaded: {len(retriever.chunks)} chunks indexed')

In [ ]:
DEMO_QUERY = 'What is the protocol for managing sepsis in a critical care unit?'
print(f'Query: {DEMO_QUERY}\n--- Retrieving...\n')

chunks = retriever.retrieve(DEMO_QUERY)

for i, c in enumerate(chunks, 1):
    print(f'[{i}] {c.section}  (score={c.score:.4f})')
    print(f'    merckmanuals.com{c.url}')
    print(f'    {c.text[:150].strip()}...\n')

**✅ Checkpoint 3:** you get `top_k` chunks, each with a section name, a similarity score, a source URL, and a text preview — and the top results are clearly about sepsis.

Expected output (illustrative):
```text
[1] Sepsis and Septic Shock  (score=0.83)
    merckmanuals.com/professional/critical-care-medicine/sepsis-and-septic-shock
    Sepsis is a clinical syndrome of life-threatening organ dysfunction...
```

## Section 4 — Generate a grounded answer

*Start state: the chat model is already running alongside the embedding model (Section 2), so we go straight to generation. We pass the retrieved chunks to the model with citation rules enforced.*

In [ ]:
from src.desktop_client import DesktopClient

llm = DesktopClient(
    base_url=os.getenv('INFERENCE_API_URL', _desktop),
    model=os.getenv('INFERENCE_MODEL', 'Qwen2.5-14B-Instruct'),
)
print(f'Inference endpoint : {llm.base_url}')
print(f'Model              : {llm.model}')

In [ ]:
result = llm.generate(question=DEMO_QUERY, chunks=chunks, max_tokens=512, temperature=0.1)

print('=' * 70)
print('ANSWER')
print('=' * 70)
print(result['answer'])
print(f"\nTokens used: {result.get('usage')}")

**✅ Checkpoint 4:** the answer is grounded in the retrieved passages, ends with a **CITATIONS** section listing the Merck sources used, and includes the medical disclaimer. If the context didn't cover the question, the model says so rather than inventing an answer.

Expected shape (illustrative):
```text
Sepsis management in critical care follows...
1. Early recognition and source control...

CITATIONS
[Sepsis and Septic Shock] — merckmanuals.com/professional/...

⚠️ Medical disclaimer: ...
```

## Section 5 — Serve and evaluate

*Start state: in a separate terminal, start the API — `uvicorn src.api:app --port 8000` — with the inference server running. Then run the cells below.*

In [ ]:
import requests

API_URL = os.getenv('API_URL', 'http://localhost:8000')
health = requests.get(f'{API_URL}/health').json()
print('Health check:')
for k, v in health.items():
    print(f'  {k:<18} {v}')

In [ ]:
BENCHMARK_QUERIES = [
    'What is the protocol for managing sepsis in a critical care unit?',
    'What are the common symptoms for appendicitis, and can it be cured via medicine?',
    'What are the effective treatments for sudden patchy hair loss on the scalp?',
    'What treatments are recommended for traumatic brain injury?',
    'What are the precautions and treatment steps for a leg fracture during a hiking trip?',
]

import time
for i, q in enumerate(BENCHMARK_QUERIES, 1):
    t0 = time.perf_counter()
    resp = requests.post(f'{API_URL}/query', json={'question': q, 'max_tokens': 512}).json()
    elapsed = int((time.perf_counter() - t0) * 1000)
    sources = [s['section'] for s in resp.get('sources', [])]
    print(f"[{i}] {q[:55]}...")
    print(f"     latency: {resp.get('latency_ms', elapsed)} ms | sources: {sources}")
    print(f"     answer : {resp.get('answer','')[:110].strip()}...\n")

In [ ]:
# Reproducible scoring — heuristic, no LLM-as-judge
import subprocess, json
import pandas as pd

subprocess.run(
    [sys.executable, os.path.join(project_root, 'eval', 'run_eval.py'),
     '--api-url', API_URL,
     '--output', os.path.join(project_root, 'eval', 'results.json'),
     '--report'],
    check=False,
)

results_path = os.path.join(project_root, 'eval', 'results.json')
if os.path.exists(results_path):
    data = json.load(open(results_path))
    print('Aggregate scores:')
    agg = {k: v for k, v in data['aggregate'].items() if k not in ('queries_scored', 'queries_failed', 'avg_latency_ms')}
    display(pd.DataFrame([agg]).T.rename(columns={0: 'score'}))
else:
    print('No results yet — make sure the API is running, then re-run this cell.')

**✅ Checkpoint 5:** `/health` returns the model name and `index_loaded: true`; all 5 benchmark queries return grounded answers with sources; and `eval/report.html` is generated with scores for groundedness, relevance, citation rate, disclaimer presence, and overall.

If any query returns an error, check that **both** the inference server (Desktop) and the API (`uvicorn`) are running.

## Extension challenges (optional)

Make it your own — each is optional and open-ended:

- **Add a topic.** Add a Merck URL to `scripts/scraper.py`, re-run scrape + `build_index.py`, and ask about it. Did retrieval surface the new content?
- **Tune `TOP_K`.** Try 3 then 8 in `.env`, re-run the eval, and compare groundedness vs. latency.
- **Swap the model.** Point `INFERENCE_MODEL` at another chat model in your Desktop catalog and compare answers on the 5 benchmarks — no code changes.
- **Bring your own corpus.** Replace the scraper with a different public dataset; the rest of the pipeline is domain-agnostic.

## Recap

You built a clinical RAG end to end — index → retrieve → generate → serve → evaluate — running entirely on your own machine. Because the documents, the embeddings, and both models stay local, nothing you index or ask ever leaves your laptop — the property that makes an approach like this workable for sensitive domains such as healthcare and legal.

**Where to go next:** [Anaconda AI Navigator](https://www.anaconda.com/docs/tools/ai-navigator/main) · [FAISS wiki](https://github.com/facebookresearch/faiss/wiki) · [Evidently docs](https://docs.evidentlyai.com) · README → *Build it yourself* for the full setup.

In [ ]:
print('Guide complete ✅')
print('\nComponents used (all served locally):')
for pkg, desc in [
    ('Anaconda Desktop', 'Local chat + embedding model servers'),
    ('FAISS',            'Dense vector similarity search'),
    ('FastAPI',          'REST API'),
    ('Evidently',        'Reproducible RAG evaluation'),
    ('Gradio',           'Interactive demo UI'),
    ('requests',         'Model API calls'),
]:
    print(f'  • {pkg:<18} {desc}')